In [2]:
# @@ENV@@ — Keras backend, visible GPU and global seed
import os

os.environ["KERAS_BACKEND"] = "tensorflow"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")

for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)


import kagglesdk.kaggle_env as kaggle_env

if not hasattr(kaggle_env, "get_web_endpoint"):
    kaggle_env.get_web_endpoint = kaggle_env.get_endpoint

import kagglehub
import keras
import keras_cv

print("GPUs:", gpus)

2026-08-16 11:25:49.703895: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1786872349.716007 3688235 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1786872349.719600 3688235 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


### **Setup**

In [ ]:
# @@CFG@@ — the knobs the post-hoc needs; split and sampling live in the frozen frames
class CFG:
    seed = 49
    image_size = (256, 256)
    batch_size = 64
    num_classes = 1
    xai_balance = "equal"   # half per class: the sample of the study
    xai_n = None            # None = the largest this split allows (140 here); a number is exploratory

keras.utils.set_random_seed(CFG.seed)

In [ ]:
# @@LIBS@@ - Common libraries used across the post-hoc pipeline.
import glob, gc, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras

In [ ]:
# @@PATHS@@ — dataset, variant and row identity are declared HERE; every path derives from them
import sys

BASE_PATH = os.environ.get("DUM_BASE", os.environ.get("ISIC24_BASE", "/mnt/disk1/home/cgiovannoni/mod-attendance"))
sys.path.insert(0, os.path.join(BASE_PATH, "src")) if os.path.join(BASE_PATH, "src") not in sys.path else None

from isic24 import paths as pth
from isic24 import xai_subset as xs

pth.DATASET, pth.VARIANT = "isic24", "enhanced"   # BRSET/Symile: change these two, nothing else
xs.ID_COL = "isic_id"                             # row identity column of this dataset

DATA_DIR = str(pth.data_dir())                    # images (hdf5) live at dataset level
DATA_PATH = str(pth.data_path())                  # metadata of this variant
ART_PATH = str(pth.art_path())
FEATURE_SPACE_PATH = os.path.join(ART_PATH, "feature_space")
RUNS_PATH = os.path.join(ART_PATH, "runs")
SUMMARIES_PATH = os.path.join(ART_PATH, "summaries")

TRAIN_META = os.path.join(DATA_PATH, "enhanced_train_metadata_clean.parquet")
TEST_META = os.path.join(DATA_PATH, "enhanced_test_metadata_clean.parquet")
TRAIN_HDF5 = os.path.join(DATA_DIR, "train-image.hdf5")
TEST_HDF5 = os.path.join(DATA_DIR, "test-image.hdf5")

print(f"dataset={pth.DATASET}/{pth.VARIANT} | id={xs.ID_COL}\n  data {DATA_PATH}\n  art  {ART_PATH}")

dataset=isic24/enhanced | id=isic_id
  data /mnt/disk1/home/cgiovannoni/mod-attendance/data/isic24/enhanced
  art  /mnt/disk1/home/cgiovannoni/mod-attendance/artifacts/isic24/enhanced


In [ ]:
# @@FRAMES@@ — load the frames the training froze; this notebook never rebuilds the split
import json as _json
import pandas as pd

validation_df = pd.read_parquet(os.path.join(ART_PATH, "evaluation_frame.parquet"))
training_df = pd.read_parquet(os.path.join(ART_PATH, "training_frame.parquet"))

_snap = _json.load(open(os.path.join(ART_PATH, "validation_snapshot.json")))
_live = [str(v) for v in xs.row_ids(validation_df)]
_frozen = [str(v) for v in _snap["row_ids"]]
if _live != _frozen:
    _lost, _new = set(_frozen) - set(_live), set(_live) - set(_frozen)
    raise RuntimeError(
        f"[FRAMES] the evaluation frame ({len(_live)} rows) is not the frozen validation "
        f"({len(_frozen)} rows): "
        + ("the same rows in a DIFFERENT ORDER." if not (_lost or _new)
           else f"{len(_lost)} missing, {len(_new)} new.")
        + " Every val_index in the XAI artifacts is a position into the frozen frame.")

print(f"[FRAMES] evaluation {validation_df.shape}, {int(validation_df.target.sum())} positive, "
      f"prevalence {validation_df.target.mean():.4f} | training {training_df.shape}")

[FRAMES] evaluation (791, 81), 70 positive, prevalence 0.0885 | training (4305, 81)


In [ ]:
# @@CUSTOM_LAYERS@@ — register the shared Keras layers, so load_model needs no custom_objects
import sys, os
_src = os.path.join(BASE_PATH, "src")
if _src not in sys.path:
    sys.path.insert(0, _src)

from isic24.mod_attendance import (
    ConstantScale, ScalarMultiply, LearnableAlpha, FusionWithGamma, GateEntropyLoss)

[FS GUARD] functions defined. `feature_space` not in scope yet — call assert_feature_space_canonical(feature_space) after cell 16, and assert_fs_matches_model(model, feature_space) after each load_model.


In [ ]:
# @@PARAMETERS@@ — canonical study dirs, budgets and the run selection of the loops below
CANONICAL = {
    "vanilla":             os.path.join(RUNS_PATH, "vanilla", "20260814-102807"),
    "fixed_scaling":       os.path.join(RUNS_PATH, "fixed_scaling", "20260812-001020"),
    "learnable_scaling":   os.path.join(RUNS_PATH, "learnable_scaling", "20260812-063607"),
    "loss_regularization": os.path.join(RUNS_PATH, "loss_regularization", "20260812-180911"),
    "controlled_fusion":   os.path.join(RUNS_PATH, "controlled_fusion", "20260812-052145"),
    "adaptive_gating":     os.path.join(RUNS_PATH, "adaptive_gating", "20260812-060120"),
}

N_CANONICAL_RUNS = 132   # 22 settings x 6 seeds
_runs = {i: sorted(glob.glob(os.path.join(d, "*", "*", "config.json"))) for i, d in CANONICAL.items()}
_spaces = {c: json.load(open(c)).get("feature_space_file") for cs in _runs.values() for c in cs}
for _i, _cs in _runs.items():
    print(f"  {_i:>19} {os.path.basename(CANONICAL[_i]):<16} {len(_cs):>3} runs  {{{_cs and _spaces[_cs[0]]}}}")
assert sum(map(len, _runs.values())) == N_CANONICAL_RUNS and len(set(_spaces.values())) == 1 \
    and None not in _spaces.values(), "not 132 runs against one FeatureSpace: read the lines above"


cf_study_dir = CANONICAL["controlled_fusion"]   # gamma sweep writes here
ag_study_dir = CANONICAL["adaptive_gating"]     # gate diagnostics write here

FAITH_BUDGETS = (0.05, 0.10, 0.20, 0.40, 0.60)   # q points the model is re-scored at (GPU)
Q_MAX_REPORT = [0.20, 0.60]                      # integration cutoffs, re-read from saved curves
FAITH_N_SAMPLES = 128                            # SHAP cost ceiling only; WHO is explained is CFG.xai_*

FAITH_ONLY_IMPL = None    # e.g. ["vanilla", "adaptive_gating"]; None = every canonical family
FAITH_MAX_RUNS  = None    # e.g. 5 for a smoke test
FAITH_OVERWRITE = False   # True = recompute and overwrite the curves a run already holds
FAITH_DRY_RUN   = False   # True = list what would run, load no models

cf_combined = pd.read_csv(os.path.join(cf_study_dir, "combined_summary.csv"))
ag_combined = pd.read_csv(os.path.join(ag_study_dir, "combined_summary.csv"))
print(f"canonical {len(CANONICAL)} families | cf_combined {cf_combined.shape} | ag_combined {ag_combined.shape}")

              vanilla 20260814-102807    6 runs  {feature_space/20260810-192622/feature_space.keras}
        fixed_scaling 20260812-001020   54 runs  {feature_space/20260810-192622/feature_space.keras}
    learnable_scaling 20260812-063607   30 runs  {feature_space/20260810-192622/feature_space.keras}
  loss_regularization 20260812-180911   30 runs  {feature_space/20260810-192622/feature_space.keras}
    controlled_fusion 20260812-052145    6 runs  {feature_space/20260810-192622/feature_space.keras}
      adaptive_gating 20260812-060120    6 runs  {feature_space/20260810-192622/feature_space.keras}
canonical 6 families | cf_combined (6, 20) | ag_combined (6, 19)


In [ ]:
# @@HDF5@@ — the image store the evaluation frame reads from
import h5py

training_validation_hdf5 = h5py.File(TRAIN_HDF5, 'r')

In [ ]:
# @@FEATURE_SPACE@@ — the space the checkpoints were trained with; the feature spec comes from it
from isic24.xai_driver import feature_space_fingerprint

FEATURE_SPACE_DIR = os.path.join(FEATURE_SPACE_PATH, "20260810-192622")

feature_space = keras.saving.load_model(os.path.join(FEATURE_SPACE_DIR, "feature_space.keras"))
assert isinstance(feature_space, keras.utils.FeatureSpace), (
    f"{FEATURE_SPACE_DIR} deserializes to {type(feature_space).__name__}, not a FeatureSpace")

feature_order = list(feature_space.features)   # declared order; SHAP reads attributions by it
FEAT_COLS = feature_order

# The identity the runs record: which file, and which adapt state. Both read off the space.
FEATURE_SPACE_FILE = os.path.relpath(
    os.path.join(FEATURE_SPACE_DIR, "feature_space.keras"), ART_PATH)
ENCODED_DIM, FEATURE_SPACE_FP = feature_space_fingerprint(feature_space)

_missing = [c for c in FEAT_COLS if c not in validation_df.columns]
assert not _missing, f"the frozen frames do not carry {len(_missing)} model features: {_missing}"
print(f"FeatureSpace {FEATURE_SPACE_FILE} | {len(FEAT_COLS)} features -> {ENCODED_DIM} encoded dims"
      f" | adapt fp {FEATURE_SPACE_FP[:16]}")

I0000 00:00:1786614156.568353 2864410 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22634 MB memory:  -> device: 0, name: Quadro RTX 6000, pci bus id: 0000:17:00.0, compute capability: 7.5


FeatureSpace 20260810-192622 | 39 features -> 130 encoded dims


In [ ]:
# @@DATASET_BUILDERS@@ — decoder and dataset builder; the post-hoc never shuffles nor augments
from tqdm import tqdm

def build_decoder(with_labels=True, target_size=CFG.image_size):
    def decode_image(inp):
        image = tf.io.decode_jpeg(inp["images"])
        image = tf.image.resize(image, size=target_size, method="area")
        image = tf.cast(image, tf.float32) / 255.0
        image = tf.reshape(image, [*target_size, 3])
        return {"images": image, "features": inp["features"]}

    def decode_with_labels(inp, label):
        label = tf.reshape(tf.cast(label, tf.float32), [CFG.num_classes])
        return decode_image(inp), label

    return decode_with_labels if with_labels else decode_image


def build_dataset(row_ids, hdf5, features, labels=None, batch_size=None, cache=True):
    images = [hdf5[row_id][()] for row_id in tqdm(row_ids, desc="Loading images")]
    inputs = {"images": images, "features": features}
    slices = (inputs, labels) if labels is not None else inputs

    ds = tf.data.Dataset.from_tensor_slices(slices)
    ds = ds.map(build_decoder(labels is not None), num_parallel_calls=tf.data.AUTOTUNE,
                deterministic=True)
    if cache:
        ds = ds.cache()
    if batch_size is not None:
        ds = ds.batch(batch_size)
    return ds


def _to_feature_dict(df):
    feat_dict = {}
    for col in df.columns:
        series = df[col]
        feat_dict[col] = (series.to_numpy() if pd.api.types.is_numeric_dtype(series.dtype)
                          else series.astype(str).to_numpy())
    return feat_dict

In [ ]:
# @@BUILD_DATASETS@@ — the evaluation set, decoded once and encoded in frame order
validation_ds = build_dataset(
    xs.row_ids(validation_df), training_validation_hdf5,
    _to_feature_dict(validation_df[FEAT_COLS]), validation_df.target.to_numpy(),
    batch_size=CFG.batch_size,
).map(lambda x, y: ({"images": x["images"], "features": feature_space(x["features"])}, y),
      num_parallel_calls=tf.data.AUTOTUNE, deterministic=True).prefetch(tf.data.AUTOTUNE)

_labels = np.concatenate([np.asarray(y).reshape(-1) for _, y in validation_ds])
assert np.array_equal(_labels, validation_df.target.to_numpy()), (
    f"the dataset yields {len(_labels)} rows against the frame's {len(validation_df)}, or the same "
    f"rows in another order: every val_index is a position into the frame")
print(f"[EVAL] {len(_labels)} rows in frame order, {len(FEAT_COLS)} features -> {ENCODED_DIM} encoded dims")

Loading images: 100%|██████████| 791/791 [00:00<00:00, 5524.17it/s]


[EVAL] 791 rows, 39 features -> 130 encoded dims


In [ ]:
# @@ENCODED_NAMES@@ — names of the encoded columns in FeatureSpace order
from keras.layers import StringLookup, Discretization, Normalization

encoded_feature_names = []
feature_dim_sizes = {}

for name in feature_order:
    preproc = feature_space.preprocessors[name]

    # 1) Categorical features: StringLookup → one-hot of size vocabulary_size
    if isinstance(preproc, StringLookup):
        vocab = preproc.get_vocabulary()
        # each token gets its own encoded dim
        for token in vocab:
            encoded_feature_names.append(f"{name}__{token}")
        d = len(vocab)

    # 2) Discretized float (age_approx): Discretization → one-hot over bins
    elif isinstance(preproc, Discretization):
        boundaries = preproc.get_config().get("bin_boundaries", [])
        n_bins = len(boundaries) + 1
        for b in range(n_bins):
            encoded_feature_names.append(f"{name}_bin_{b}")
        d = n_bins

    # 3) Normalized numeric or raw float: 1 dim
    elif isinstance(preproc, Normalization):
        encoded_feature_names.append(name)
        d = 1

    else:
        # "float" (already normalized) or any other scalar
        encoded_feature_names.append(name)
        d = 1

    feature_dim_sizes[name] = d

# sanity check
print("per-feature dim sizes:", feature_dim_sizes)
print("len(encoded_feature_names):", len(encoded_feature_names))
print("feature_space encoded dim:", feature_space.get_encoded_features().shape[-1])

assert len(encoded_feature_names) == feature_space.get_encoded_features().shape[-1]

per-feature dim sizes: {'sex': 4, 'anatom_site_general': 7, 'tbp_tile_type': 3, 'tbp_lv_location': 22, 'combined_anatomical_site': 22, 'attribution': 8, 'age_approx': 32, 'tbp_lv_nevi_confidence': 1, 'clin_size_long_diam_mm': 1, 'tbp_lv_areaMM2': 1, 'tbp_lv_area_perim_ratio': 1, 'tbp_lv_color_std_mean': 1, 'tbp_lv_deltaLBnorm': 1, 'tbp_lv_minorAxisMM': 1, 'lesion_size_ratio': 1, 'lesion_shape_index': 1, 'hue_contrast': 1, 'luminance_contrast': 1, 'lesion_color_difference': 1, 'border_complexity': 1, '3d_position_distance': 1, 'perimeter_to_area_ratio': 1, 'lesion_visibility_score': 1, 'symmetry_border_consistency': 1, 'color_consistency': 1, 'size_age_interaction': 1, 'hue_color_std_interaction': 1, 'lesion_severity_index': 1, 'shape_complexity_index': 1, 'color_contrast_index': 1, 'log_lesion_area': 1, 'mean_hue_difference': 1, 'std_dev_contrast': 1, 'color_shape_composite_index': 1, '3d_lesion_orientation': 1, 'overall_color_difference': 1, 'symmetry_perimeter_interaction': 1, 'compr

In [ ]:
# @@SETUP_XAI@@ — bind the session context, then adopt the XAI sample once for every engine
from isic24 import xai_faithfulness as xf
from isic24.xai_faithfulness import (
    explanation_faithfulness, stratified_table, pool_by_family, discover_curve_files,
    verify_reintegration, _bootstrap_ci, _load_shap_values,
)
from isic24.xai_driver import run_posthoc_xai
from isic24.xai_engines import _finalize_plot

xf.configure(
    xf.XaiContext(
        feat_cols=FEAT_COLS, training_df=training_df, validation_df=validation_df,
        validation_ds=validation_ds, feature_space=feature_space,
        to_feature_dict=_to_feature_dict, encoded_feature_names=encoded_feature_names,
        feature_space_file=FEATURE_SPACE_FILE, feature_space_fingerprint=FEATURE_SPACE_FP,
    ),
    FAITH_N_SAMPLES=FAITH_N_SAMPLES,   # SHAP cost ceiling only; who gets explained is the sample below
    FAITH_BUDGETS=tuple(FAITH_BUDGETS),
    EVAL_IS_CANONICAL=True,            # the frozen evaluation frame is the only set this notebook reads
)

# One central setting per family: the comparison set behind every table and figure.
CENTRAL_SETTINGS = {"vanilla": "baseline", "fixed_scaling": "a0p500",
                    "learnable_scaling": "init_a0p500", "loss_regularization": "lam1e-03",
                    "controlled_fusion": "base", "adaptive_gating": "base"}

# One sample per session, on DISTINCT lesions, shared by SHAP, Grad-CAM, LORE and faithfulness.
xai_subset_idx, xai_subset_info = xs.adopt(
    validation_df, n=CFG.xai_n, balance=CFG.xai_balance, seed=0, snapshot_dir=ART_PATH)
XAI_SUBSET_FP = xai_subset_info["fingerprint"]

[sample] 140 DISTINCT lesions (balance='equal', seed=0)
         per class: {0: 70, 1: 70}   prevalence 0.500
         unique available: {0: 721, 1: 70}
         check: 140/140 distinct ids OK
         fingerprint: unique-equal-s0-n140-4e93cb16fae6032a
         active on: faithfulness, lore, shap
         snapshot: /mnt/disk1/home/cgiovannoni/mod-attendance/artifacts/isic24/enhanced/subset.unique-equal.n140.seed0.json
         restore the historical prefix: xs.adopt(validation_df, balance=None)


In [ ]:
# @@STOP@@ — the setup ends here, everything below explains checkpoints
raise StopIteration("setup complete — the cells below run one at a time")

StopIteration: 

In [ ]:
# @@FEATURE_PROVENANCE@@ 1/2 — where the tabular features come from
from isic24.feature_provenance import engineered_provenance, verify_engineered

BUILD_NB = os.path.join(BASE_PATH, "notebooks", "data_preproc", "isic24_create_dataset.ipynb")
BUILD_FUNCS = ("feature_engineering", "clean_df")
SOURCE_META = os.path.join(DATA_DIR, "train-metadata.csv")
CONTEXT_COLS = ["sex", "age_approx", "anatom_site_general", "tbp_lv_location",
                "combined_anatomical_site", "attribution", "tbp_tile_type"]
MODEL_OUTPUT_COLS = ["tbp_lv_nevi_confidence"]
IMAGE_DERIVED_COLS = [c for c in FEAT_COLS if c not in CONTEXT_COLS]
NOT_CANDIDATE_COLS = ["isic_id", "patient_id", "lesion_id", "target", "image_type",
                      "copyright_license", "tbp_lv_dnn_lesion_confidence", "iddx_full",
                      "iddx_1", "iddx_2", "iddx_3", "iddx_4", "iddx_5"]

BUILD_SOURCE, PROVENANCE = engineered_provenance(BUILD_NB, BUILD_FUNCS)
natural_df = pd.read_parquet(TRAIN_META)
built_cols = set(natural_df.columns) - set(pd.read_csv(SOURCE_META, nrows=0).columns)
assert built_cols == set(PROVENANCE), (
    f"the build adds columns the map does not name: {sorted(built_cols - set(PROVENANCE))} | "
    f"named without being added: {sorted(set(PROVENANCE) - built_cols)}")
ENGINEERED_FEAT_COLS = [c for c in FEAT_COLS if c in PROVENANCE]
RELEASE_FEAT_COLS = [c for c in FEAT_COLS if c not in PROVENANCE]
RAW_TBP_COLS = sorted(c for c in natural_df.columns
                      if c.startswith("tbp_lv_") and c not in NOT_CANDIDATE_COLS)

differing = verify_engineered(natural_df, BUILD_SOURCE, PROVENANCE, BUILD_FUNCS)
assert not differing.any(), (
    f"engineered columns that are not what they read: {differing[differing > 0].to_dict()}")
leaked = sorted(set(FEAT_COLS) & set(NOT_CANDIDATE_COLS))
assert not leaked, f"identity or diagnostic columns among the model features: {leaked}"

used_cols = [c for c in natural_df.columns if c in FEAT_COLS]
dropped_cols = [c for c in natural_df.columns
                if c not in FEAT_COLS and c not in NOT_CANDIDATE_COLS]
reached = set().union(*(PROVENANCE[c] for c in ENGINEERED_FEAT_COLS))
reexpressed = [c for c in dropped_cols if c in reached]
absent = [c for c in dropped_cols if c not in reached]

print(f"[provenance] {len(FEAT_COLS)} features = {len(CONTEXT_COLS)} context + "
      f"{len(IMAGE_DERIVED_COLS)} image-derived, of which model output: {MODEL_OUTPUT_COLS}")
print(f"[variant]    {len(RELEASE_FEAT_COLS)} straight from the source metadata + "
      f"{len(ENGINEERED_FEAT_COLS)} engineered, every column the build adds named and "
      f"recomputed from its own sources over {len(natural_df)} rows")
print(f"[census]     {len(natural_df.columns)} metadata columns = {len(NOT_CANDIDATE_COLS)} not "
      f"candidates + {len(used_cols)} used + {len(dropped_cols)} dropped")
print(f"[dropped]    {len(reexpressed)} re-expressed inside an engineered feature, "
      f"{len(absent)} reaching the model in no form: {absent}")

resampled_df = pd.concat([training_df, validation_df])
for col in ("attribution", "tbp_tile_type"):
    for label, frame in (("natural", natural_df), ("resampled", resampled_df)):
        p = frame.groupby(col)["target"].mean()
        print(f"[{col:14s} {label:9s}] {len(p)} levels, prevalence "
              f"{p.min():.4f}..{p.max():.4f}, ratio {p.max() / max(p.min(), 1e-9):.1f}x")

: 

In [ ]:
# @@FEATURE_PROVENANCE@@ 2/2 — how much of the tabular AUC is site shortcut, and how much is
# feature engineering rather than information. Gradient boosting on the same split, CPU ~2 min.

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

def _split_encoded(cols):
    both = pd.concat([training_df[cols], validation_df[cols]])
    for c in both.columns:
        if not pd.api.types.is_numeric_dtype(both[c]):
            both[c] = both[c].astype("category").cat.codes
    both = both.astype("float32")
    return both.iloc[:len(training_df)], both.iloc[len(training_df):]

SUBSETS = {
    "attribution alone": ["attribution"],
    "context only": CONTEXT_COLS,
    "image-derived only": IMAGE_DERIVED_COLS,
    "all features": FEAT_COLS,
    "all minus attribution/tile": [c for c in FEAT_COLS
                                   if c not in ("attribution", "tbp_tile_type")],
    "release cols, as we use them": RELEASE_FEAT_COLS,
    "release cols, every raw TBP": sorted(set(RELEASE_FEAT_COLS) | set(RAW_TBP_COLS)),
    "engineered cols only": ENGINEERED_FEAT_COLS,
}
y_tr, y_va = training_df["target"].to_numpy(), validation_df["target"].to_numpy()
for name, cols in SUBSETS.items():
    X_tr, X_va = _split_encoded(cols)
    p = HistGradientBoostingClassifier(random_state=0).fit(X_tr, y_tr).predict_proba(X_va)[:, 1]
    print(f"[tab-only] {name:28s} n_feat {len(cols):2d}  AUC {roc_auc_score(y_va, p):.4f}")


In [ ]:
# @@LOGIT_DECOMP@@ — exact logit decomposition: z = bias + image term + tabular term, no residual
# Loads one checkpoint per run and does a single forward pass over the adopted sample; the guard raises if the split does not reconstruct the prediction.

from isic24 import xai_logit as xlog

LOGIT_ALL_RUNS  = False         # False = the six central settings | True = every canonical run
LOGIT_OVERWRITE = False         # True = re-decompose the runs that already hold a per-sample table

_lmaster = pd.read_csv(os.path.join(SUMMARIES_PATH, "master_results_rows.csv"))
_central = None if LOGIT_ALL_RUNS else CENTRAL_SETTINGS
_by_run, _by_family, _per_sample = xlog.decompose_study(
    _lmaster, ART_PATH, subset_idx=xai_subset_idx, central=_central, overwrite=LOGIT_OVERWRITE)

print("\n=== image share of the logit, by family ===")
print(_by_family.to_string(index=False))

# Three ways to ask "how much does the model use the image", side by side.
_m = _lmaster if _central is None else _lmaster[
    [bool(_central.get(i) == s) for i, s in zip(_lmaster.impl_name, _lmaster.setting_tag)]]
_cmp = (_by_run.groupby("impl_name")["mean_image_abs_share_descriptive"].mean().rename("logit_share")
        .to_frame()
        .join(_m.groupby("impl_name")[["img_drop_auc", "tab_drop_auc", "base_auc"]].mean()))
_cmp["ablation_share"] = _cmp.img_drop_auc / (_cmp.img_drop_auc + _cmp.tab_drop_auc)
print("\n=== three measures of reliance ===")
print(_cmp[["logit_share", "ablation_share", "img_drop_auc", "tab_drop_auc"]].round(3).to_string())
print(f"Spearman(logit_share, ablation_share) over {len(_cmp)} families: "
      f"{_cmp.logit_share.corr(_cmp.ablation_share, method='spearman'):+.3f}")

# Aggregating over both classes hides the asymmetry the strata already show.
print("\n=== per class: image share and target-aligned contribution ===")
print(_by_run.groupby("impl_name")[xlog.CLASS_COLS].mean().round(3).to_string())

# Consistency with the June vanilla decomposition (128 samples, same baselines).
_june = os.path.join(ART_PATH, "diagnostics", "vanilla_logit_contributions",
                     "20260604-210304", "vanilla_logit_contributions_by_seed.csv")
if os.path.exists(_june):
    _j = pd.read_csv(_june)["mean_image_abs_share_descriptive"].mean()
    _v = _by_run.loc[_by_run.impl_name == "vanilla", "mean_image_abs_share_descriptive"].mean()
    print(f"\n[check] vanilla image share — June n=128: {_j:.3f} | this sample n={len(xai_subset_idx)}: {_v:.3f}")

  [ablations] logit decomposition: 10 unimodal endpoint runs excluded from the aggregate
[1/30]              vanilla   baseline share 0.717  delta 1.2e-07
[2/30]              vanilla   baseline share 0.545  delta 1.2e-07
[3/30]              vanilla   baseline share 0.649  delta 1.5e-07
[4/30]              vanilla   baseline share 0.683  delta 6.0e-08
[5/30]              vanilla   baseline share 0.669  delta 1.2e-07
[6/30]        fixed_scaling     a0p500 share 0.756  delta 1.2e-07
[7/30]        fixed_scaling     a0p500 share 0.570  delta 1.2e-07
[8/30]        fixed_scaling     a0p500 share 0.686  delta 1.2e-07
[9/30]        fixed_scaling     a0p500 share 0.667  delta 1.2e-07
[10/30]        fixed_scaling     a0p500 share 0.729  delta 1.2e-07
[11/30]    learnable_scaling init_a0p500 share 0.656  delta 1.8e-07
[12/30]    learnable_scaling init_a0p500 share 0.650  delta 1.2e-07
[13/30]    learnable_scaling init_a0p500 share 0.673  delta 1.2e-07
[14/30]    learnable_scaling init_a0p500 share

In [ ]:
# @@GAMMA_ATTENDANCE@@ — attendance against the fusion bias, re-derived from the saved decomposition
_g_by_run, _g_by_family = xlog.gamma_curve(_per_sample, ART_PATH, master=_lmaster)

_neutral = _g_by_run[_g_by_run.delta == 0.0].set_index("run_id")["mean_image_abs_share_descriptive"]
_diff = (_by_run.set_index("run_id")["mean_image_abs_share_descriptive"] - _neutral).abs().max()
print(f"[check] delta=0 reproduces the measured decomposition: max |diff| = {_diff:.2e}")

print("\n=== image share of the logit against the fusion bias ===")
print(_g_by_family.pivot(index="impl_name", columns="gamma_tab",
                         values="mean_image_abs_share_descriptive__mean").round(3).to_string())

print("\n=== sample AUC against the fusion bias (adopted XAI sample, NOT the validation set) ===")
print(_g_by_family.pivot(index="impl_name", columns="gamma_tab",
                         values="sample_auc__mean").round(4).to_string())

_peak = _g_by_run.loc[_g_by_run.groupby(["impl_name", "seed"])["sample_auc"].idxmax()]
print("\n=== where each run's sample AUC peaks, by family ===")
print(_peak.groupby("impl_name")["delta"].agg(["mean", "std", "count"]).round(3).to_string())


In [ ]:
# @@REPORT_FIGURES@@ — the sec. 3.6 figure, the one a table cannot carry (CPU, no checkpoint)
# Self-contained: paths and central settings come from the modules, so a fresh kernel is enough.
from isic24 import xai_figures as xfig

_arms = xfig.arms()
display(_arms.groupby("impl_name")[["del_exp", "del_rand", "del_gain",
                                    "ins_exp", "ins_rand", "ins_gain"]].mean().round(3))


In [ ]:
# @@CAM_OVERLAP@@ — do two families look in the same places more than two seeds do?
from isic24 import xai_metrics as xmet

_ov = xmet.cam_overlap_table(pd.read_csv(os.path.join(SUMMARIES_PATH, "master_results_rows.csv")),
                             ART_PATH, central=xs.CENTRAL_SETTINGS)
display(_ov.groupby("pair_kind")[["iou_half", "iou_top"]].agg(["median", "mean", "count"]).round(3))
print("the counts above are PAIRS: this table describes, it does not test")

_contrast = pd.concat([xmet.seed_level_contrast(_ov, c, "cross_family", "seed_null").assign(iou=c)
                       for c in ("iou_half", "iou_top")], ignore_index=True)
print("\nCross-family against the seed null, with the SEED as the unit")
display(_contrast.set_index("iou").round(4))
print(f"{int(_contrast.n_seeds.max())} seeds put the floor at p = {_contrast.p_floor.max():.3f};")
print("positive diff = two FAMILIES overlap more than two seeds of one family do.")

In [ ]:
# @@SAMPLE_AUDIT@@ — do the four engines of a run explain the SAME rows? (CPU)
# Nesting is legitimate (LORE explains 48 of the 144); a partial overlap is two samples, one name.
_pop, _pop_ok = xs.sample_audit(pd.read_csv(os.path.join(SUMMARIES_PATH, "master_results_rows.csv")),
                                ART_PATH)


              runs  n_min  n_max  knows_rows
kind                                        
faithfulness   110    128    128           0
gradcam_img    110    144    144         110
lore_sa        110     48     48         110
shap_tab       110    144    144         110
   110 runs  [SAMPLE] not one population — faithfulness: n=128 vs 144, rows unknown — it predates `val_index`, so nesting cannot be shown

[SAMPLE AUDIT] every population nested in the sample: FAIL


## Post-hoc diagnostics — gate faithfulness, calibration, gamma sweep

In [ ]:
# @@DIAG_HELPERS@@ — AUC over a dataset

def eval_auc(model, ds):
    m = keras.metrics.AUC()
    for x, y in ds:
        m.update_state(y, model(x, training=False))
    return float(m.result().numpy())

In [ ]:
# @@GATE_DIAGNOSTICS@@ — gate weights, ablation and calibration per adaptive-gating run (GPU)
from sklearn.metrics import roc_auc_score
from isic24 import xai_driver as xd, xai_logit as xl

_TRAINED = ["base_auc", "img_drop_auc", "tab_drop_auc", "brier", "ece"]
_NEEDED = {"run_id", "run_dir", "setting_tag", "seed", "entropy_weight", *_TRAINED}
assert _NEEDED <= set(ag_combined.columns) and len(ag_combined) == 6, (
    f"ag_combined is the study aggregate, read for entropy_weight and for the metrics training "
    f"already measured: {len(ag_combined)} rows, missing "
    f"{sorted(_NEEDED - set(ag_combined.columns))} — a whitespace-padded header is how one of "
    f"the canonical studies would land here")

all_diag = []
for _, row in ag_combined.iterrows():
    ag_run_dir = xs.run_dir_abs(row["run_dir"], ART_PATH)
    ag_model, _ = xd.load_run_model(ag_run_dir)
    gate = xl.gate_per_sample(ag_model, validation_ds)

    _seen = {"base_auc": float(roc_auc_score(gate.target, gate.prediction)),
             "brier": float(np.mean((gate.prediction - gate.target) ** 2))}
    _off = {k: (v, float(row[k])) for k, v in _seen.items() if abs(v - float(row[k])) > 5e-3}
    assert not _off, (f"{row['run_id']}: the checkpoint disagrees with what training recorded, "
                      f"{_off} as (post-hoc, training) — same weights on the same frame")

    diag = dict(run_id=row["run_id"], setting_tag=row["setting_tag"], seed=int(row["seed"]),
                entropy_weight=float(row["entropy_weight"]), run_dir=ag_run_dir,
                **{c: float(row[c]) for c in _TRAINED}, **xl.gate_summary(gate))
    os.makedirs(os.path.join(ag_run_dir, "diagnostics"), exist_ok=True)
    with open(os.path.join(ag_run_dir, "diagnostics", "adaptive_gating_diagnostics.json"), "w") as f:
        json.dump(diag, f, indent=2)
    all_diag.append(diag)

    plots_dir = os.path.join(ag_run_dir, "plots")
    os.makedirs(plots_dir, exist_ok=True)

    plt.figure(figsize=(7, 4))
    plt.hist(gate.w_img, bins=30, alpha=0.6, label="w_img")
    plt.hist(gate.w_tab, bins=30, alpha=0.6, label="w_tab")
    plt.xlabel("gate weight"); plt.ylabel("count")
    plt.title(f"Adaptive gating: gate weights (seed {int(row['seed'])}, n={diag['n']})")
    plt.legend(); plt.grid(alpha=0.3)
    _finalize_plot(plots_dir, "gate_weight_distribution.png")

    plt.figure(figsize=(7, 4))
    _bars = ["base_auc", "img_drop_auc", "tab_drop_auc"]
    _vals = [diag[c] for c in _bars]
    plt.bar(_bars, _vals, color=["#4c72b0", "#dd8452", "#55a868"])
    for j, v in enumerate(_vals):
        plt.text(j, v, f"{v:.3f}", ha="center", va="bottom", fontsize=9)
    plt.ylabel("AUC"); plt.grid(alpha=0.3, axis="y")
    plt.title(f"Adaptive gating: modality ablation (seed {int(row['seed'])})")
    _finalize_plot(plots_dir, "modality_ablation_auc_drop.png")

    _edges = np.linspace(0.0, 1.0, 16)
    _conf, _obs = [], []
    for _lo, _hi in zip(_edges[:-1], _edges[1:]):
        _m = (gate.prediction >= _lo) & ((gate.prediction <= _hi) if _hi >= 1.0
                                         else (gate.prediction < _hi))
        if _m.any():
            _conf.append(float(gate.prediction[_m].mean()))
            _obs.append(float(gate.target[_m].mean()))
    plt.figure(figsize=(5, 5))
    plt.plot([0, 1], [0, 1], "--", color="gray", label="perfect")
    plt.plot(_conf, _obs, marker="o", label="model")
    plt.xlabel("mean predicted probability"); plt.ylabel("observed frequency")
    plt.title(f"Calibration (Brier={diag['brier']:.3f}, ECE={diag['ece']:.3f})")
    plt.legend(); plt.grid(alpha=0.3)
    _finalize_plot(plots_dir, "calibration_reliability.png")

    keras.backend.clear_session()
    gc.collect()

ag_diag_df = pd.DataFrame(all_diag)
_diag_csv = os.path.join(ag_study_dir, "adaptive_gating_diagnostics.csv")
xs.assert_same_population(_diag_csv, ag_diag_df.run_id)
ag_diag_df.to_csv(_diag_csv, index=False)
display(ag_diag_df)

In [ ]:
# @@CF_GAMMA_SWEEP@@ — post-hoc gamma sweep over every controlled-fusion run (GPU)
# gamma_tab > 1 enhances tabular and penalises image; gamma_tab < 1 does the opposite.
DELTAS = np.round(np.arange(-0.5, 0.501, 0.125), 3)
POSTHOC_SETTINGS = []
for d in DELTAS:
    POSTHOC_SETTINGS.append({
        "tag": f"d{d:+.3f}".replace(".", "p").replace("+", "plus").replace("-", "minus"),
        "delta": float(d),
        "gamma_img": float(1.0 - d),
        "gamma_tab": float(1.0 + d),
        "mode": "tab_enhance_img_penalize" if d > 0 else ("tab_penalize_img_enhance" if d < 0 else "neutral"),
    })

from isic24 import xai_driver as xd

posthoc_rows = []
for _, row in cf_combined.iterrows():
    cf_run_dir = xs.run_dir_abs(row["run_dir"], ART_PATH)
    best_model, _ = xd.load_run_model(cf_run_dir)
    fusion_layer = best_model.get_layer("fusion_with_gamma")

    run_rows = []
    for s in POSTHOC_SETTINGS:
        fusion_layer.gamma_img.assign(s["gamma_img"])
        fusion_layer.gamma_tab.assign(s["gamma_tab"])
        auc_val = eval_auc(best_model, validation_ds)

        rec = {
            "run_id": row["run_id"],
            "setting_tag": row["setting_tag"],
            "seed": int(row["seed"]),
            "run_dir": cf_run_dir,
            "posthoc_id": s["tag"],
            "delta": s["delta"],
            "gamma_img": s["gamma_img"],
            "gamma_tab": s["gamma_tab"],
            "mode": s["mode"],
            "val_auc": auc_val,
            "base_model_name": row["model_name"],
        }
        run_rows.append(rec)
        posthoc_rows.append(rec)

    posthoc_df = pd.DataFrame(run_rows).sort_values("gamma_tab").reset_index(drop=True)
    diagnostics_dir = os.path.join(cf_run_dir, "diagnostics")
    os.makedirs(diagnostics_dir, exist_ok=True)
    posthoc_df.to_csv(os.path.join(diagnostics_dir, "posthoc_gamma_summary.csv"), index=False)

    plots_dir = os.path.join(cf_run_dir, "plots")
    os.makedirs(plots_dir, exist_ok=True)

    plt.figure(figsize=(7, 4))
    plt.plot(posthoc_df["gamma_tab"], posthoc_df["val_auc"], marker="o")
    plt.axvline(1.0, linestyle="--", linewidth=1)
    plt.xlabel("gamma_tab")
    plt.ylabel("val_auc")
    plt.title(f"Controlled fusion post-hoc: val_auc vs gamma_tab (seed {int(row['seed'])})")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, "posthoc_val_auc_vs_gamma_tab.png"), dpi=300)
    plt.close()

    plt.figure(figsize=(7, 4))
    plt.scatter(posthoc_df["gamma_img"], posthoc_df["gamma_tab"], c=posthoc_df["val_auc"])
    for _, r in posthoc_df.iterrows():
        plt.text(r["gamma_img"], r["gamma_tab"], f"{r['val_auc']:.3f}", fontsize=8)
    plt.xlabel("gamma_img")
    plt.ylabel("gamma_tab")
    plt.title(f"Controlled fusion post-hoc sweep (seed {int(row['seed'])})")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, "posthoc_gamma_sweep_scatter.png"), dpi=300)
    plt.close()

    keras.backend.clear_session()
    gc.collect()

posthoc_all_df = pd.DataFrame(posthoc_rows).sort_values(["seed", "gamma_tab"]).reset_index(drop=True)
posthoc_all_df.to_csv(os.path.join(cf_study_dir, "controlled_fusion_posthoc_gamma_summary.csv"), index=False)

posthoc_agg = (
    posthoc_all_df.groupby(["delta", "gamma_img", "gamma_tab"], dropna=False)["val_auc"]
    .agg(["mean", "std", "count"])
    .reset_index()
)
posthoc_agg.to_csv(os.path.join(cf_study_dir, "controlled_fusion_posthoc_gamma_aggregated.csv"), index=False)
display(posthoc_all_df.head(20))
display(posthoc_agg)

## Explanation faithfulness (deletion / insertion AUC)

`faithfulness_gain = deletion_auc(explanation) − deletion_auc(random)`, computed for `tabular` (SHAP), `image` (Grad-CAM) and `joint`. It reuses the SHAP arrays already on disk, so the GradientExplainer is not re-fitted. The three cells below define the loop, run it over every trained run and aggregate the result.


In [ ]:
# @@FAITH_LOOP@@ — deletion/insertion on the adopted sample, both tabular attributors (GPU)
FAITH_OUT_DIR = os.path.join(RUNS_PATH, "_faithfulness")
os.makedirs(FAITH_OUT_DIR, exist_ok=True)
_fs_declared = os.path.relpath(os.path.join(FEATURE_SPACE_DIR, "feature_space.keras"), ART_PATH)


_plan = []
for _impl, _study in CANONICAL.items():
    if FAITH_ONLY_IMPL and _impl not in FAITH_ONLY_IMPL:
        continue
    for _sp in sorted(glob.glob(os.path.join(_study, "*", "*", "single_run_summary.csv"))):
        _row, _rd = pd.read_csv(_sp).iloc[0], os.path.dirname(_sp)
        _fs_run = json.load(open(os.path.join(_rd, "config.json"))).get("feature_space_file")
        assert _fs_run == _fs_declared, f"{_rd} was trained against {_fs_run}, not {_fs_declared}"
        _js = {"aligned": os.path.join(_rd, "xai", "faithfulness", "faithfulness.json"),
               "gradient": os.path.join(_rd, "xai", "faith_gradient", "faithfulness.json")}
        _plan.append({"impl_name": _impl, "setting_tag": _row.setting_tag, "seed": int(_row.seed),
                      "best_epoch": int(_row.best_epoch), "best_val_auc": float(_row.best_val_auc),
                      "run_dir": _rd, "js": _js,
                      "ckpt": os.path.join(_rd, "checkpoints", f"best_{_row.model_name}.keras"),
                      "todo": [k for k, f in _js.items() if FAITH_OVERWRITE or not os.path.exists(f)]})

faith_plan = pd.DataFrame(_plan).head(FAITH_MAX_RUNS or len(_plan))
print(f"[FAITH] {len(faith_plan)} canonical runs | {int((faith_plan.todo.map(len) > 0).sum())} with work left "
      f"| sample {XAI_SUBSET_FP} (n={len(xai_subset_idx)}), the same rows for every run")
if FAITH_DRY_RUN:
    display(faith_plan.drop(columns=["ckpt", "js"]))
    raise SystemExit("FAITH_DRY_RUN=True: no models loaded.")

# The gradient ranking is READ from disk, never recomputed: without SHAP on the adopted sample
# there is nothing to rank with, and finding out one model at a time costs a night.
_no_shap = [r.run_dir for r in faith_plan.itertuples()
            if "gradient" in r.todo and not glob.glob(
                os.path.join(r.run_dir, "xai", "shap_tab", "epoch_*", "*.npz"))]
assert not _no_shap, (f"{len(_no_shap)} runs carry no SHAP on this sample: run @@SUBSET@@ 2/4 "
                      f"with the SHAP engine first — e.g. {_no_shap[0]}")


def _grad_cols(d):   # the gradient contrast, prefixed so both attributors sit in one row
    return {f"grad_{k}": v for k, v in d.items() if k.startswith("faithfulness_") or k == "tab_ranking"}


faith_rows = []
for _r in faith_plan.itertuples():
    _t0 = time.time()
    if _r.todo:
        keras.backend.clear_session()
        model = keras.models.load_model(_r.ckpt, compile=False, safe_mode=False)
        assert_fs_matches_model(model, feature_space, _r.run_dir)
    if "aligned" in _r.todo:
        _raw, _im, _ = xf._build_faith_subset()
        _phi, _ = xf.aligned_tab_attributions(model, _raw, _im, background="baseline")
        res = explanation_faithfulness(model, _r.run_dir, _r.best_epoch,
                                       methods=("tabular", "image", "joint"),
                                       tab_imp_override=np.abs(_phi),
                                       save=True, verbose=False, allow_clobber=FAITH_OVERWRITE)
    else:
        res = {k: v for k, v in json.load(open(_r.js["aligned"])).items()
               if k not in ("curves", "q_grid")}
    if "gradient" in _r.todo:
        res.update(_grad_cols(explanation_faithfulness(
            model, _r.run_dir, _r.best_epoch, methods=("tabular", "joint"),
            out_dir=os.path.dirname(_r.js["gradient"]), save=True, verbose=False,
            allow_clobber=FAITH_OVERWRITE)))
    else:
        res.update(_grad_cols(json.load(open(_r.js["gradient"]))))
    if _r.todo:
        res["faith_seconds"] = round(time.time() - _t0, 2)
        keras.backend.clear_session(); gc.collect()
    faith_rows.append({**{c: getattr(_r, c) for c in
                          ("impl_name", "setting_tag", "seed", "best_val_auc", "run_dir")},
                       **{k: v for k, v in res.items() if not isinstance(v, (list, dict))}})
    print(f"  {'+'.join(_r.todo) or 'reuse':>17} {_r.impl_name:>18}/{_r.setting_tag:<14} "
          f"seed {_r.seed:<4} del={res['faithfulness_gain']:+.4f} ({res.get('faith_seconds', '-')}s)")

faith_df = pd.DataFrame(faith_rows)
_out_csv = os.path.join(FAITH_OUT_DIR, "combined_faithfulness.csv")
if os.path.exists(_out_csv):          # a partial pass must not truncate the registry
    _prev = pd.read_csv(_out_csv)
    faith_df = pd.concat([_prev[~_prev.run_dir.isin(faith_df.run_dir)], faith_df], ignore_index=True)
faith_df.to_csv(_out_csv, index=False)
print(f"[FAITH] {faith_df.shape} -> {_out_csv}")
display(faith_df.head(20))

In [ ]:
# @@MASTER_TABLE@@ — one row per canonical run, from the per-run summaries -> <artifacts>/summaries/
os.makedirs(SUMMARIES_PATH, exist_ok=True)
master = xs.build_registry(CANONICAL, n_expected=N_CANONICAL_RUNS)
master["run_dir"] = master.run_dir.map(lambda p: xs.run_dir_abs(p, ART_PATH))

_rows_csv = os.path.join(SUMMARIES_PATH, "master_results_rows.csv")
xs.assert_same_population(_rows_csv, master.run_id)

_faith_csv = os.path.join(RUNS_PATH, "_faithfulness", "combined_faithfulness.csv")
_fcols = ["tab_ranking", "faithfulness_gain", "faithfulness_tabular_del_gain",
          "faithfulness_image_del_gain"]
if os.path.exists(_faith_csv):
    _f = pd.read_csv(_faith_csv)
    _f["run_dir"] = _f.run_dir.map(lambda p: xs.run_dir_abs(p, ART_PATH))
    master = master.merge(_f[["run_dir"] + [c for c in _fcols if c in _f.columns]]
                          .drop_duplicates("run_dir"), on="run_dir", how="left")
    _n_faith = int(master.faithfulness_gain.notna().sum())
    assert _n_faith, f"{_faith_csv} matches no canonical run: it is another population's"
    print(f"[REGISTRY] faithfulness merged on {_n_faith}/{len(master)} runs")
else:
    print(f"[REGISTRY] no combined_faithfulness.csv: the registry carries no faithfulness "
          f"column yet — run @@FAITH_LOOP@@, then this cell again")

_metrics = [c for c in ["best_val_auc", "brier", "ece", "modality_balance", "faithfulness_gain",
            "faithfulness_tabular_del_gain", "faithfulness_image_del_gain"]
            if c in master.columns]
summary = master.groupby(["impl_name", "setting_tag"], dropna=False)[_metrics].agg(
    ["mean", "std", "count"])
master.to_csv(_rows_csv, index=False)
summary.to_csv(os.path.join(SUMMARIES_PATH, "master_results_summary.csv"))
print(f"[REGISTRY] {len(master)} runs, {master.impl_name.nunique()} families, "
      f"{master.setting_tag.nunique()} settings -> {SUMMARIES_PATH}")
display(summary)

[REGISTRY] no combined_faithfulness.csv: the registry carries no faithfulness column yet — run @@FAITH_LOOP@@, then this cell again
[REGISTRY] 132 runs, 6 families, 21 settings -> /mnt/disk1/home/cgiovannoni/mod-attendance/artifacts/isic24/enhanced/summaries


best_val_auc                     brier  \
                                        mean       std count      mean   
impl_name           setting_tag                                          
adaptive_gating     base            0.948887  0.008588     6  0.045685   
controlled_fusion   base            0.944343  0.006366     6  0.044632   
fixed_scaling       a0p000          0.909839  0.010292     6  0.055596   
                    a0p125          0.931907  0.008432     6  0.048710   
                    a0p250          0.936167  0.014005     6  0.047387   
                    a0p375          0.939170  0.008147     6  0.046743   
                    a0p500          0.942453  0.008042     6  0.046948   
                    a0p625          0.941132  0.008319     6  0.048432   
                    a0p750          0.945575  0.006010     6  0.045056   
                    a0p875          0.949815  0.005665     6  0.046467   
                    a1p000          0.954871  0.001763     6  0.051567   
learnable_scaling   init_a0p100     0.931844  0.010053     6  0.048548   
                    init_a0p300     0.936918  0.014133     6  0.047868   
                    init_a0p500     0.940443  0.009857     6  0.047659   
                    init_a0p700     0.945889  0.004285     6  0.045377   
                    init_a0p900     0.949049  0.003079     6  0.042863   
loss_regularization lam0e+00        0.942715  0.005194     6  0.047426   
                    lam1e-02        0.942395  0.009309     6  0.045513   
                    lam1e-03        0.943243  0.011656     6  0.042801   
                    lam1e-04        0.941959  0.008083     6  0.045446   
                    lam1e-05        0.943075  0.008315     6  0.045446   
vanilla             baseline        0.941388  0.008728     6  0.045417   

                                                      ece                  \
                                      std count      mean       std count   
impl_name           setting_tag                                             
adaptive_gating     base         0.004933     6  0.042823  0.016666     6   
controlled_fusion   base         0.003049     6  0.028137  0.004948     6   
fixed_scaling       a0p000       0.007173     6  0.046304  0.011083     6   
                    a0p125       0.003362     6  0.037967  0.004239     6   
                    a0p250       0.003299     6  0.040190  0.008464     6   
                    a0p375       0.002057     6  0.038197  0.009749     6   
                    a0p500       0.003358     6  0.035242  0.003598     6   
                    a0p625       0.003520     6  0.043741  0.023584     6   
                    a0p750       0.004157     6  0.044977  0.016643     6   
                    a0p875       0.004162     6  0.058396  0.030550     6   
                    a1p000       0.001379     6  0.075070  0.004776     6   
learnable_scaling   init_a0p100  0.002772     6  0.037755  0.003925     6   
                    init_a0p300  0.003160     6  0.034176  0.005508     6   
                    init_a0p500  0.004287     6  0.037658  0.009493     6   
                    init_a0p700  0.001384     6  0.046102  0.019791     6   
                    init_a0p900  0.004592     6  0.043880  0.036152     6   
loss_regularization lam0e+00     0.003102     6  0.032355  0.008094     6   
                    lam1e-02     0.006231     6  0.039054  0.019405     6   
                    lam1e-03     0.003806     6  0.031449  0.010532     6   
                    lam1e-04     0.004554     6  0.037041  0.012057     6   
                    lam1e-05     0.001164     6  0.038541  0.014827     6   
vanilla             baseline     0.007316     6  0.043776  0.024353     6   

                                modality_balance                      
                                            mean           std count  
impl_name           setting_tag                                       
adaptive_gating     base                0.

In [ ]:
# @@STRATA_SECTION@@ 1/3 — build the full faithfulness table from the saved curves
from isic24 import xai_faithfulness as xf

FAITH_STRATA_SIGNED = True    # include the signed variants

_files = xf.discover_curve_files(ART_PATH, include_signed=FAITH_STRATA_SIGNED)
print("[curve] coverage by source:")
display(_files.groupby("source").agg(run=("run_id", "nunique"), file=("npz_path", "size")))

# Guard: the mean of the per-sample gains must reproduce the aggregate saved in the JSONs.
_probs = xf.verify_reintegration(_files[_files.source == "canonical"].npz_path.iloc[0])
print("[guard] verify_reintegration:", "OK (<1e-9)" if not _probs else f"** FAILED ** {_probs}")

strata = xf.stratified_table(ART_PATH, files=_files, q_maxes=tuple(Q_MAX_REPORT), verbose=False)
pooled = xf.pool_by_family(strata)

_p1 = os.path.join(RUNS_PATH, "_faithfulness", "faithfulness_strata.csv")
_p2 = os.path.join(RUNS_PATH, "_faithfulness", "faithfulness_strata_pooled.csv")
xs.assert_same_population(_p1, strata.run_id)
strata.to_csv(_p1, index=False)
pooled.to_csv(_p2, index=False)
print(f"\n[wrote] {_p1}   ({len(strata)} rows: one per run x source x cell)")
print(f"[wrote] {_p2}   ({len(pooled)} rows, pooled by family)")
print("\nBoth are derived: this cell regenerates them at any time, they are not canonical.")

In [ ]:
# @@STRATA_SECTION@@ 2/3 — summary table: one row per family, every attributor side by side
# Deletion, stratum "all", column gain_norm (per unit of budget); positive = faithful.
_h = pooled[(pooled.stratum == "all") & (pooled.direction == "del")].copy()
_h["attributore"] = np.where(
    _h.source == "canonical", "SHAPgrad:" + _h.method,
    np.where(_h.source == "aligned", "perm:tabular", "signed:" + _h["mode"]))

_piv = _h.pivot_table(index="impl_name", columns=["attributore", "q_max"], values="gain_norm")
print("gain_norm — deletion, all samples (positive = faithful)")
display(_piv.round(3))

# How many runs sit behind each number.
print("\nruns per cell (n_runs):")
display(_h.pivot_table(index="impl_name", columns="attributore", values="n_runs", aggfunc="max"))

print("attributors: SHAPgrad:* = GradientExplainer on the named branch | "
      "perm:tabular = logit-aligned Permutation | signed:* = the same, evidence for/against.")

In [ ]:
# @@STRATA_SECTION@@ 3/3 — strata: by class, and by confidence x correctness
# Shown at the low cutoff, the clean regime; at full budget the metric is OOD-dominated.
STRATA_SHOW = ["all", "class_pos", "class_neg",
               "sure_correct", "sure_wrong", "unsure_correct", "unsure_wrong"]
_QLOW = min(Q_MAX_REPORT)

_s = pooled[(pooled.direction == "del") & (pooled.q_max == _QLOW)
            & (pooled.stratum.isin(STRATA_SHOW))]

for _src, _mode, _meth, _title in [
        ("aligned", "abs", "tabular", "aligned Permutation — tabular, |attribution|"),
        ("canonical", "abs", "joint", "SHAP GradientExplainer — joint"),
        ("aligned_signed", "target", "tabular", "signed Permutation — true class (`target`)")]:
    _t = _s[(_s.source == _src) & (_s["mode"] == _mode) & (_s.method == _meth)]
    if not len(_t):
        print(f"\n[{_title}] no rows — source not on disk.")
        continue
    print(f"\n{_title}   (deletion, q<={_QLOW}, gain_norm)")
    display(_t.pivot_table(index="impl_name", columns="stratum", values="gain_norm")
              .reindex(columns=[c for c in STRATA_SHOW if c in set(_t.stratum)]).round(3))

print("\nEmpty cells are withheld on purpose below MIN_CELL_N; column `n` in `strata` "
      "gives the real sample count.")

## LORE explanations — regenerate them with the `lore_sa` engine

The only GPU-bound part of the post-hoc; everything else runs on the saved curves. These cells need the **full** setup (data, `FeatureSpace`, `xf.configure`), because the driver loads checkpoints and perturbs real features.

Measured cost: ~0.67 min per sample, so ~21 min per run at n=32.


In [ ]:
# @@LORE_SECTION@@ — aggregate the per-sample LORE CSVs the run above wrote (CPU)
# Two artifact paths exist; the newer xai/lore_sa/ wins per run, the historical one only fills gaps.
from isic24.run_ids import run_id_from_artifact

_nuovi = sorted(glob.glob(os.path.join(RUNS_PATH, "**", "xai", "lore_sa", "epoch*",
                                       "lore_sa_persample_*.csv"), recursive=True))
_storici = sorted(glob.glob(os.path.join(RUNS_PATH, "**", "xai", "lore", "epoch*",
                                         "tabular_sa", "lore_sa_persample_*.csv"),
                            recursive=True))
_visti = {run_id_from_artifact(f) for f in _nuovi}
_solo_storici = [f for f in _storici if run_id_from_artifact(f) not in _visti]
_lore_files = _nuovi + _solo_storici
print(f"[LORE] {len(_lore_files)} runs: {len(_nuovi)} from xai/lore_sa/ (new), "
      f"{len(_solo_storici)} from xai/lore/.../tabular_sa/ (historical, n=32)"
      + (f" — {len(_storici) - len(_solo_storici)} historical files skipped, the run "
         f"already has the new one" if len(_storici) != len(_solo_storici) else ""))
if _solo_storici and _nuovi:
    print("       [warn] the two sources differ in n and prevalence: compare the ordering, "
          "not the levels, until all runs are redone.")

if not _lore_files:
    print("No `lore_sa` artifact yet: run `@@SUBSET@@ 4/4`, which writes xai/lore_sa/.")
else:
    # Run identity comes from `isic24.run_ids`, never from re-parsing the path by hand.
    _master = pd.read_csv(os.path.join(SUMMARIES_PATH, "master_results_rows.csv"))
    _meta = {str(r.run_id): (r.impl_name, r.setting_tag) for _, r in _master.iterrows()}

    _rows, _unmatched = [], []
    for f in _lore_files:
        _rid = run_id_from_artifact(f)
        if _rid not in _meta:
            _unmatched.append(f"{_rid}   ({f})")
            continue
        d = pd.read_csv(f)
        d["run_id"] = _rid
        d["impl_name"], d["setting_tag"] = _meta[_rid]
        _rows.append(d)
    if _unmatched:
        raise RuntimeError("LORE artifacts absent from the canonical registry:\n  "
                           + "\n  ".join(_unmatched))

    L = pd.concat(_rows, ignore_index=True)
    L["covered"] = L["neigh_n_classes"] >= 2

    fam = L.groupby(["impl_name", "setting_tag"]).agg(
        n_run=("run_id", "nunique"),
        n_campioni=("i", "size"),
        lore_coverage=("covered", "mean"),                       # the attendance signal
        neigh_pos_rate=("neigh_pos_rate", "mean"),
        neigh_unique_frac=("neigh_unique_frac", "mean"),
        rule_len=("rule_len", "mean"),
        fid_cv_bacc=("fid_cv_bacc", "mean"),
    )
    # Fidelity only means something where the neighbourhood has two classes.
    fam["n_covered"] = L.groupby(["impl_name", "setting_tag"])["covered"].sum()
    fam["fid_cv_bacc_covered"] = (L[L.covered]
                              .groupby(["impl_name", "setting_tag"])["fid_cv_bacc"].mean())
    print("\nLORE explanation quality by family "
          "(`lore_coverage` = how far the tabular branch drives the decision)")
    display(fam.round(3))


    # Class-wise on the explanations, with the RUN as the unit: all runs share one sample,
    # so pooling rows across runs would inflate the p-value.
    cw = (L.groupby(["impl_name", "setting_tag", "true_label"])
            .agg(n=("i", "size"), lore_coverage=("covered", "mean"),
                 neigh_pos_rate=("neigh_pos_rate", "mean"),
                 rule_len=("rule_len", "mean"), fid_cv_bacc=("fid_cv_bacc", "mean"))
            .unstack("true_label"))
    print("\nthe same quantities split by true class (0 = benign, 1 = malignant)")
    display(cw.round(3))
    cw.reset_index().to_csv(
        os.path.join(RUNS_PATH, "_faithfulness", "lore_sa_quality_by_family_class.csv"),
        index=False)

    print("\nthe between-class contrast, tested across runs:")
    _per = L.groupby(["run_id", "true_label"])["covered"].mean().unstack().dropna()
    if len(_per) >= 2:
        xs.print_across_runs(xs.across_runs(_per[0], _per[1], "benign", "malignant"))
    else:
        print("  at least 2 runs with both classes are needed.")

    _out = os.path.join(RUNS_PATH, "_faithfulness", "lore_sa_quality_by_family.csv")
    fam.reset_index().to_csv(_out, index=False)
    print("wrote", _out)

## Distinct-lesion XAI sample

`pos_sample=5.0` uses `replace=True`, so 364 unique positives become 1820 rows and the same lesion is explained several times: the per-run confidence intervals are optimistic. The cells below build a sample of distinct lesions, check that its fingerprint round-trips through the saved SHAP arrays, measure how far the numbers move, and recompute SHAP, Grad-CAM, aligned faithfulness and LORE on it.

The saved npz store `explained_indices` as **positions**, so reusing them under a different sample would pair the SHAP values of some rows with the features of other lesions — silently, and with plausible numbers. That is what the fingerprint guard in `_load_shap_values` prevents.


In [ ]:
# @@SETUP_XAI@@ — bind the session context, then adopt the XAI sample once for every engine
from isic24 import xai_faithfulness as xf
from isic24.xai_faithfulness import (
    explanation_faithfulness, stratified_table, pool_by_family, discover_curve_files,
    verify_reintegration, _bootstrap_ci, _load_shap_values,
)
from isic24.xai_driver import run_posthoc_xai
from isic24.xai_engines import _finalize_plot

xf.configure(
    xf.XaiContext(
        feat_cols=FEAT_COLS, training_df=training_df, validation_df=validation_df,
        validation_ds=validation_ds, feature_space=feature_space,
        to_feature_dict=_to_feature_dict, encoded_feature_names=encoded_feature_names,
        feature_space_file=FEATURE_SPACE_FILE, feature_space_fingerprint=FEATURE_SPACE_FP,
    ),
    FAITH_N_SAMPLES=FAITH_N_SAMPLES,   # SHAP cost ceiling only; who gets explained is the sample below
    FAITH_BUDGETS=tuple(FAITH_BUDGETS),
)

# One central setting per family: the comparison set behind every table and figure.
CENTRAL_SETTINGS = {"vanilla": "baseline", "fixed_scaling": "a0p500",
                    "learnable_scaling": "init_a0p500", "loss_regularization": "lam1e-03",
                    "controlled_fusion": "base", "adaptive_gating": "base"}

# One sample per session, on DISTINCT lesions, shared by SHAP, Grad-CAM, LORE and faithfulness.
xai_subset_idx, xai_subset_info = xs.adopt(
    validation_df, n=CFG.xai_n, balance=CFG.xai_balance, seed=0, snapshot_dir=ART_PATH)
XAI_SUBSET_FP = xai_subset_info["fingerprint"]

In [ ]:
# @@SUBSET@@ 2/4 — Grad-CAM, SHAP and LORE on the adopted sample, resumable per run (GPU)

from isic24 import xai_driver as xd
import gc, keras, time, os, numpy as np, pandas as pd

JOBS_TO_RUN = ["gradcam", "shap", "lore"]          # ["gradcam", "shap", "lore"]
ONLY_IMPL   = None                                 # e.g. ["vanilla"]; None = every family
ONLY_SEEDS  = None                                 # e.g. [47]; None = every seed
PER_FAMILY  = None                                 # e.g. 1; None = every run
LORE_N      = 48
DRY_RUN     = False
LORE_ON_CPU = True

xd.LORE_SA_DEVICE = "/CPU:0" if LORE_ON_CPU else None

# Guard: with no adopted sample the engines would write into the canonical folders.
assert xd.shap_subdir() == "shap_tab" and xd.gradcam_subdir() == "gradcam_img", (
    "no sample adopted in this kernel: run the @@SETUP_XAI@@ cell, which calls `xs.adopt(...)`.\n"
    "  Without it SHAP and Grad-CAM would overwrite the canonical prefix artifacts.")

_m = pd.read_csv(os.path.join(SUMMARIES_PATH, "master_results_rows.csv"))
# Seeds are numbers in the master: coerce, so ["47"] does not silently select nothing.
if ONLY_SEEDS is not None: ONLY_SEEDS = [int(_s) for _s in ONLY_SEEDS]
if ONLY_IMPL:  _m = _m[_m.impl_name.isin([str(_x) for _x in ONLY_IMPL])]
if ONLY_SEEDS: _m = _m[_m.seed.astype(int).isin(ONLY_SEEDS)]
if PER_FAMILY: _m = _m.groupby("impl_name", sort=False).head(int(PER_FAMILY))
# Guard: an empty selection is not a successful pass, it is a pass that never started.
_all = pd.read_csv(os.path.join(SUMMARIES_PATH, "master_results_rows.csv"))
if not len(_m):
    raise ValueError(
        f"\n  The filters select no run.\n"
        f"    ONLY_IMPL  = {ONLY_IMPL}      allowed: {sorted(_all.impl_name.unique())}\n"
        f"    ONLY_SEEDS = {ONLY_SEEDS}      allowed: {sorted(_all.seed.unique())}\n"
        f"    PER_FAMILY = {PER_FAMILY}\n"
        f"  (ONLY_IMPL takes a FAMILY name, not an engine: engines are chosen with "
        f"JOBS_TO_RUN.)")
_m = _m.reset_index(drop=True)
_RD = {r.run_id: xs.run_dir_abs(r.run_dir, ART_PATH)
       for _, r in _m.iterrows()}
print(f"selected {len(_m)} runs: {sorted(_m.impl_name.unique())} | seeds {sorted(_m.seed.unique())}")

_lore_idx = xs.lore_subset(validation_df.target.values, xai_subset_idx, LORE_N)

# The plan, printed before the GPU is taken.
print(f"XAI sample: {XAI_SUBSET_FP}")
print(f"  SHAP and Grad-CAM on {len(xai_subset_idx)} lesions | LORE on {len(_lore_idx)} "
      f"({LORE_N//2} pos / {LORE_N//2} neg)\n")
_plan, _tot = {}, 0.0
for _j in JOBS_TO_RUN:
    _cfg = xs.JOBS[_j]
    _n = len(_lore_idx) if _j == "lore" else len(xai_subset_idx)
    _exp = _lore_idx if _j == "lore" else np.asarray(xai_subset_idx)
    _todo, _done, _redo = xs.resume_verified(_RD, _cfg["kind"], XAI_SUBSET_FP, _exp,
                                             task=_cfg["task"])
    _work = _todo + _redo               # `_redo` = stamped, but with a different sample
    _sec = _cfg["sec_per_sample"]
    _h = (len(_work) * _n * _sec / 3600) if _sec else None
    _tot += _h or 0
    _plan[_j] = (_work, set(_redo), _cfg, _n)
    print(f"  {_j:8s} -> xai/{_cfg['kind']:12s} done {len(_done):3d} | to do {len(_todo):3d}"
          + (f" | TO REDO {len(_redo):3d} (other sample)" if _redo else "")
          + (f" | ~{_h:.1f} h ({_n*_sec/60:.1f} min/run)" if _h else " | cost NOT MEASURED"))
print(f"\n  TOTAL ESTIMATE: ~{_tot:.1f} h" + ("  (+ Grad-CAM, not measured)"
      if any(xs.JOBS[j]["sec_per_sample"] is None for j in JOBS_TO_RUN) else ""))
print("  Interrupt at any time: re-running resumes where it stopped.")

if DRY_RUN:
    print("\n*** DRY RUN — nothing executed. Set DRY_RUN=False to run. ***")
else:
    _t0, _orphans = time.time(), []
    for _j in JOBS_TO_RUN:
        _todo, _redo_set, _cfg, _n = _plan[_j]
        if not _todo:
            continue
        print(f"\n===== {_j.upper()} — {len(_todo)} runs =====", flush=True)
        if _j == "lore":
            xd.LORE_SA_KWARGS["max_explain_samples"] = len(_lore_idx)
            xd.LORE_SA_KWARGS["subset_idx"] = _lore_idx
            print(f"  (LORE on {xd.LORE_SA_DEVICE or 'the default device = GPU'})",
                  flush=True)
        for _k, _rid in enumerate(_todo):
            _rd, _row = _RD[_rid], _m[_m.run_id == _rid].iloc[0]
            _t1 = time.time()
            try:
                # allow_clobber only for runs to redo; elsewhere the write guard stays armed.
                _o = xd.run_posthoc_xai(_rd, engines=_cfg["engines"],
                                        best_epoch=int(_row.best_epoch),
                                        best_val_auc=float(_row.best_val_auc),
                                        overwrite=True, verbose=False,
                                        allow_clobber=(_rid in _redo_set))
                # run_posthoc_xai does NOT raise on engine failure: it returns "failed: ...",
                # so a failed run must never be stamped as done.
                _bad = {k: v for k, v in (_o or {}).items() if str(v) != "ok"}
                if _bad or not _o:
                    print(f"  [FAIL] {_rid}: {_bad or 'no engine ran'}", flush=True)
                    continue
                xs.mark_done(_rd, _cfg["kind"], XAI_SUBSET_FP, task=_cfg["task"],
                             payload={"run_id": _rid, "impl_name": _row.impl_name,
                                      "setting_tag": _row.setting_tag, "n": int(_n),
                                      "result": str(_o)})
                print(f"  [{_k+1}/{len(_todo)}]{' REDONE' if _rid in _redo_set else ''} "
                      f"{_rid}: {_o}  "
                      f"({(time.time()-_t1)/60:.1f} min | {(time.time()-_t0)/60:.0f} tot)",
                      flush=True)
            except RuntimeError as _e:
                if "WRITE GUARD" in str(_e):
                    # Artifacts from an attempt killed before stamping: report, never clobber.
                    _orphans.append((_j, _rid, os.path.join(_rd, "xai", _cfg["kind"])))
                    print(f"  [ORPHANS] {_rid}: unstamped {_j} artifacts", flush=True)
                else:
                    print(f"  [FAIL] {_rid}: {type(_e).__name__}: {_e}", flush=True)
            except Exception as _e:
                print(f"  [FAIL] {_rid}: {type(_e).__name__}: {_e}", flush=True)
            finally:
                keras.backend.clear_session(); gc.collect()
        if _j == "lore":
            xd.LORE_SA_KWARGS.pop("subset_idx", None)

    print(f"\n===== closing ({(time.time()-_t0)/60:.0f} min) =====")
    for _j in JOBS_TO_RUN:
        _cfg = xs.JOBS[_j]
        _exp = _lore_idx if _j == "lore" else np.asarray(xai_subset_idx)
        _todo, _done, _redo = xs.resume_verified(_RD, _cfg["kind"], XAI_SUBSET_FP, _exp,
                                                 task=_cfg["task"])
        print(f"  {_j:8s} ok {len(_done)}/{len(_m)} on the right sample"
              + (f" — {len(_todo)+len(_redo)} left, re-run this cell"
                 if (_todo or _redo) else "  [complete]"))
    if _orphans:
        print(f"\n[WARN] {len(_orphans)} runs have ORPHAN artifacts, written by an attempt "
              f"killed before stamping. The write guard refuses to overwrite them.\n"
              f"  To redo them, delete the folder and re-run:")
        for _j, _rid, _p in _orphans[:10]:
            print(f"    rm -rf {_p}    # {_j} {_rid}")


In [ ]:
# @@SUBSET@@ 3/4 — aligned signed tabular faithfulness on the distinct-lesion sample (GPU)
# ~3.6 min/run at n=144; writes into <run_dir>/xai/shap_tab/ and resumes per run.
from isic24 import xai_driver as xd
import gc, keras, time, os, pandas as pd

STRAT_ONLY_IMPL  = None         # None = every family
STRAT_ONLY_SEEDS = None         # e.g. [47]; None = every seed
STRAT_PER_FAMILY = None         # e.g. 1 for one run per family; None = every run
STRAT_DRY_RUN    = False        # False to run

_m = pd.read_csv(os.path.join(SUMMARIES_PATH, "master_results_rows.csv"))
if STRAT_ONLY_IMPL:  _m = _m[_m.impl_name.isin(STRAT_ONLY_IMPL)]
if STRAT_ONLY_SEEDS: _m = _m[_m.seed.isin(STRAT_ONLY_SEEDS)]
if STRAT_PER_FAMILY: _m = _m.groupby("impl_name", sort=False).head(int(STRAT_PER_FAMILY))
_m = _m.reset_index(drop=True)

# run_id -> run_dir, re-anchored on ART_PATH: the master carries the paths of another machine.
_RD = {r.run_id: xs.run_dir_abs(r.run_dir, ART_PATH)
       for _, r in _m.iterrows()}

# 1.5 s/sample, measured with the four signed modes; changing SIGNED_MODES changes it.
SEC_PER_SAMPLE = 1.5
_todo, _done = xs.resume_report(_RD, "shap_tab", XAI_SUBSET_FP)
_est = len(_todo) * len(xai_subset_idx) * SEC_PER_SAMPLE
print(f"[subset] {len(_m)} runs selected | already done {len(_done)} | to do {len(_todo)}")
print(f"[subset] {len(xf.SIGNED_MODES)} modes x {len(xai_subset_idx)} samples "
      f"-> estimate {_est/60:.0f} min ({_est/3600:.1f} h)  [{_est/60/max(len(_todo),1):.1f} min/run]")
if _done:
    print(f"        (resume active: the {len(_done)} already done with fingerprint {XAI_SUBSET_FP} are skipped)")
if STRAT_DRY_RUN:
    print("\n*** DRY RUN — nothing executed. Set STRAT_DRY_RUN=False to run. ***")
else:
    _t0 = time.time()
    for _i, _r in _m.iterrows():
        _rd = _RD[_r.run_id]
        if xs.is_done(_rd, "shap_tab", XAI_SUBSET_FP):
            continue
        try:
            _model, _ = xd.load_run_model(_rd)
            # Returns (results, phi): unpack it, or mark_done is never reached.
            _o, _ = xf.aligned_signed_faithfulness(
                _model, _rd, int(_r.best_epoch), subset_idx=xai_subset_idx,
                verbose=False, q_maxes=tuple(Q_MAX_REPORT))
            # The stamp is written only here, once the run is complete: that makes it resumable.
            xs.mark_done(_rd, "shap_tab", XAI_SUBSET_FP,
                         {"run_id": _r.run_id, "impl_name": _r.impl_name,
                          "setting_tag": _r.setting_tag, "n": len(xai_subset_idx),
                          "gains": {k: v for k, v in _o.items() if isinstance(v, float)}})
            print(f"[{_i+1}/{len(_m)}] {_r.run_id}: ok  ({(time.time()-_t0)/60:.0f} min total)", flush=True)
        except Exception as _e:
            print(f"[FAIL] {_r.run_id}: {type(_e).__name__}: {_e}", flush=True)
        finally:
            keras.backend.clear_session(); gc.collect()
    _todo, _done = xs.resume_report(_RD, "shap_tab", XAI_SUBSET_FP)
    print(f"\n[subset] done {len(_done)}/{len(_m)} in {(time.time()-_t0)/60:.0f} min"
          + (f" — {len(_todo)} left, re-run this cell to resume" if _todo else ""))

    # Cross-family report, no GPU. `central` is not optional: a family mean over all 45
    # fixed_scaling rows, endpoints included, means nothing.
    if not _todo:
        xs.report(pd.read_csv(os.path.join(SUMMARIES_PATH, "master_results_rows.csv")),
                  ART_PATH, kinds=("shap_tab",), central=CENTRAL_SETTINGS,
                  encoded_feature_names=encoded_feature_names)


In [ ]:
# @@SUBSET@@ 4/4 — LORE on half the adopted sample, resumable per run (GPU)
from isic24 import xai_driver as xd
import gc, keras, time

# These filters keep the last selection used: re-read them before launching.
STRAT_LORE_ONLY_IMPL  = None        # None = every family
STRAT_LORE_ONLY_SEEDS = None        # None = every seed
STRAT_LORE_ONLY_TAG   = list(CENTRAL_SETTINGS.values())
STRAT_LORE_PER_FAMILY = 1
STRAT_LORE_DRY_RUN    = False
STRAT_LORE_N          = 48

_m = pd.read_csv(os.path.join(SUMMARIES_PATH, "master_results_rows.csv"))
if STRAT_LORE_ONLY_IMPL:  _m = _m[_m.impl_name.isin(STRAT_LORE_ONLY_IMPL)]
if STRAT_LORE_ONLY_SEEDS: _m = _m[_m.seed.isin(STRAT_LORE_ONLY_SEEDS)]
if STRAT_LORE_ONLY_TAG:   _m = _m[_m.setting_tag.isin(STRAT_LORE_ONLY_TAG)]
if STRAT_LORE_PER_FAMILY: _m = _m.groupby("impl_name", sort=False).head(int(STRAT_LORE_PER_FAMILY))
_m = _m.reset_index(drop=True)
_RD = {r.run_id: xs.run_dir_abs(r.run_dir, ART_PATH)
       for _, r in _m.iterrows()}

_lore_idx = xs.lore_subset(validation_df.target.values, xai_subset_idx, STRAT_LORE_N)
_lore_y = validation_df.target.values[_lore_idx]
print(f"[subset-lore] sample: {len(_lore_idx)} lesions, "
      f"{int(_lore_y.sum())} pos / {int((1-_lore_y).sum())} neg  (balanced)")

xd.LORE_SA_KWARGS["max_explain_samples"] = len(_lore_idx)

_todo, _done = xs.resume_report(_RD, "lore_sa", XAI_SUBSET_FP)
print(f"[subset-lore] {len(_m)} runs | already done {len(_done)} | to do {len(_todo)}")
MIN_PER_SAMPLE_LORE = 0.67       # minutes per sample, measured
print(f"[subset-lore] {len(_lore_idx)} samples/run -> estimate "
      f"{len(_todo)*len(_lore_idx)*MIN_PER_SAMPLE_LORE/60:.1f} h "
      f"({len(_lore_idx)*MIN_PER_SAMPLE_LORE:.0f} min/run)")
# A missing family or a degenerate representative must be seen now, not three hours in.
print(f"[subset-lore] families covered: {_m.impl_name.nunique()}/6")
_deg = _m[_m.setting_tag.astype(str).str.contains("a0p000|lam0e\\+00|init_a0p100")]
if len(_deg):
    print(f"\n[warn] {len(_deg)} selected runs are DEGENERATE settings (tabular branch zeroed "
          f"or lambda=0): a valid control, not a family representative.")
    display(_deg[["impl_name", "setting_tag", "run_id"]])
display(_m[["impl_name", "setting_tag", "seed", "run_id"]])

if STRAT_LORE_DRY_RUN:
    print("\n*** DRY RUN — nothing executed. Set STRAT_LORE_DRY_RUN=False to run. ***")
else:
    _prev = xd.LORE_SA_KWARGS.get("subset_idx")
    xd.LORE_SA_KWARGS["subset_idx"] = _lore_idx
    _t0 = time.time()
    try:
        for _i, _r in _m.iterrows():
            _rd = _RD[_r.run_id]
            if xs.is_done(_rd, "lore_sa", XAI_SUBSET_FP):
                continue
            try:
                # overwrite=True only disarms the xai_status.json shortcut, which would lie
                # here; the write guard stays armed and still refuses to clobber xai/lore_sa/.
                _o = xd.run_posthoc_xai(_rd, engines=("lore",), best_epoch=int(_r.best_epoch),
                                        best_val_auc=float(_r.best_val_auc),
                                        overwrite=True, verbose=False)
                # run_posthoc_xai does NOT raise on engine failure: it returns "failed: ...",
                # so a failed run must never be stamped as done.
                _bad = {_k: _v for _k, _v in (_o or {}).items() if str(_v) != "ok"}
                if _bad or not _o:
                    print(f"[FAIL] {_r.run_id}: {_bad or 'no engine ran'}", flush=True)
                    continue
                xs.mark_done(_rd, "lore_sa", XAI_SUBSET_FP,
                             {"run_id": _r.run_id, "impl_name": _r.impl_name,
                              "setting_tag": _r.setting_tag, "n": int(len(_lore_idx)),
                              "n_pos": int(_lore_y.sum()), "result": str(_o)})
                print(f"[{_i+1}/{len(_m)}] {_r.run_id}: {_o}  "
                      f"({(time.time()-_t0)/60:.0f} min total)", flush=True)
            except Exception as _e:
                print(f"[FAIL] {_r.run_id}: {type(_e).__name__}: {_e}", flush=True)
            finally:
                keras.backend.clear_session(); gc.collect()
    finally:
        xd.LORE_SA_KWARGS["subset_idx"] = _prev   # do not leave the knob armed for other cells
    _todo, _done = xs.resume_report(_RD, "lore_sa", XAI_SUBSET_FP)
    print(f"\n[subset-lore] done {len(_done)}/{len(_m)} in {(time.time()-_t0)/60:.0f} min"
          + (f" — {len(_todo)} left, re-run to resume" if _todo else ""))
    if not _todo:
        xs.report(pd.read_csv(os.path.join(SUMMARIES_PATH, "master_results_rows.csv")),
                  ART_PATH, kinds=("lore_sa",), central=CENTRAL_SETTINGS,
                  encoded_feature_names=encoded_feature_names,
                  lore_idx={len(_lore_idx): _lore_idx})

In [ ]:
# @@SHAPE@@ — explanation-shape metrics and stability, from the artifacts on disk
import pandas as pd, os
from isic24 import xai_metrics as xmet

_m = pd.read_csv(os.path.join(SUMMARIES_PATH, "master_results_rows.csv"))

_out = xs.report(_m, ART_PATH, kinds=(), central=CENTRAL_SETTINGS,
                 encoded_feature_names=encoded_feature_names)
_shape, _stab, _fam = _out["shape"], _out["stability"], _out["by_family"]
print(f"\n[shape] {len(_shape)} runs | {len(_stab)} pairs | "
      f"{int(_shape.branch_off.sum())} runs with the tabular branch off (metrics = NaN)")

print("\nStability@k by pair kind — the pair is the unit here, and this table describes")
display(_stab.pivot_table(index="impl_name", columns="pair_kind", values="stability_at_k",
                          aggfunc=["mean", "count"]).round(3))

_contrast = xmet.seed_level_contrast(_stab, "stability_at_k", "adjacent_param",
                                     "same_param_seed", by="impl_name")
print("\nThe same contrast with the SEED as the unit — this one tests")
display(_contrast.set_index("impl_name").round(4))
print("positive diff = one step of the knob moves the top-k features LESS than the seed does;")
print(f"{int(_contrast.n_seeds.max())} seeds put the floor at p = {_contrast.p_floor.max():.3f}.")
print("Families with a single setting have no adjacent-parameter arm and no contrast.")

print("\nall metrics side by side")
display(_fam.round(3))